# Detekcja anomalii

<div style="text-align: center;"><img src=".//Images//Zebra_Among_Deer.png" alt="Wartości odstające" width="400" height="120" style="margin: 10px; "/></div>

**Detekcja anomalii** (ang. *anomaly detection*) to dziedzina uczenia maszynowego i statystyki, która skupia się na wykrywaniu rzadkich, nietypowych lub podejrzanych obserwacji w danych. Anomalie mogą oznaczać błędy, oszustwa, awarie, ataki lub inne nietypowe zjawiska.

---

Podział metod detekcji anomalii:

**a) Metody statystyczne**
Zakładają, że dane pochodzą z określonego rozkładu (np. normalnego), a anomalie to punkty istotnie od niego odbiegające.
- **Z-score** – punkty o z-score powyżej np. 3 uznaje się za odstające.
- **Grubbs' Test**, **Dixon’s Q Test** – klasyczne testy dla małych próbek.
- **Rozkład kwantylowy** (np. IQR: interquartile range).

Plusy: proste, szybkie.  
Minusy: wrażliwe na założenia o rozkładzie danych.

---

**b) Metody oparte na uczeniu maszynowym**

**Uczenie nadzorowane (jeśli są etykiety anomalii):**
- **Drzewa decyzyjne, SVM, XGBoost** – klasyfikatory trenowane na danych normalnych i anomaliach.

**Uczenie nienadzorowane (najczęściej spotykane):**
- **Isolation Forest** – losowo dzieli dane i liczy liczbę podziałów potrzebną do „odizolowania” punktu. Anomalie są izolowane szybciej.  
- **One-Class SVM** – uczy granicy obejmującej większość danych; punkty poza nią to anomalie.  
- **Autoencodery** – sieci neuronowe uczące się reprezentacji wejścia. Jeśli dany punkt ma duży błąd rekonstrukcji → może być anomalią.  
- **kNN / Local Outlier Factor (LOF)** – punkt jest anomalią, jeśli jest daleko od sąsiadów lub ich gęstość jest wyraźnie większa niż jego własna.

Plusy: nie wymagają etykiet, mogą uchwycić złożone zależności.  
Minusy: czasem trudne do interpretacji, wymagają strojenia parametrów.

---

**c) Metody oparte na progach**
- Reguły typu: „jeśli wartość > próg → anomalia”.  
- Często stosowane w systemach monitorujących (np. wartości temperatury, napięcia, opóźnień).

---

**Przykładowe zastosowania**
- **Finanse**: wykrywanie oszustw transakcyjnych.
- **Cyberbezpieczeństwo**: nietypowe logowania, ataki sieciowe.
- **Medycyna**: wykrywanie rzadkich chorób, błędów pomiarowych.
- **Inżynieria**: awarie maszyn, czujniki poza normą.
- **Produkcja**: defekty produktów.

# Przykład: zbiór `creditcard.csv` (fraudy kartowe).

<div style="text-align: center;"><img src=".//Images//cards.jpg" alt="creditcard" width="400" height="120" style="margin: 10px; "/></div>

Przykład koncentruje się na **wprowadzeniu do zbioru danych *creditcard.csv***, który jest jednym z najpopularniejszych benchmarków do detekcji oszustw finansowych (tzw. *fraud detection*) w obszarze płatności kartowych. 

---

**Tło i cel zbioru danych**

1. Kontekst biznesowy i analityczny

- **Główny cel**: Wspomagać opracowywanie i weryfikowanie algorytmów wykrywających nadużycia w transakcjach kartowych (oszustwa, tzw. *fraudy*).  
- **Znaczenie**: Oszustwa z użyciem kart płatniczych stanowią realne zagrożenie dla banków i instytucji finansowych, generując duże straty. Równocześnie – ze względu na wrażliwość i poufność danych – publikowanych jest mało otwartych, wiarygodnych zbiorów o transakcjach rzeczywistych.

2. Źródło i pochodzenie

- Współtwórcami danych są badacze z Uniwersytetu w Liège (*Université de Liège*) w Belgii – zbiór użyty był do badań i artykułów naukowych związanych z metodami wykrywania anomalii/ fraudów.

---

**Zakres i zawartość zbioru danych**

1. Rozmiar i podstawowe informacje

- **Liczba obserwacji**: Około 284 000 wierszy (konkretnie 284 807). Każdy wiersz reprezentuje pojedynczą transakcję kartową.  
- **Okres obserwacji**: Dane dotyczą około 2 dni ciągłych transakcji (48 godzin).  
- **Format**: Plik CSV, w którym każda kolumna odpowiada konkretnej cesze (feature) lub atrybutowi transakcji.  

Z uwagi na realne pochodzenie danych (transakcje rzeczywistych klientów), oryginalne surowe cechy zostały ukryte (przekształcone), aby zapewnić anonimowość i bezpieczeństwo.  

2. Struktura kolumn

Zbiór zawiera 31 kolumn:  
* **`Time`** – liczba sekund, która upłynęła od pierwszej transakcji w zbiorze do momentu dokonania danej transakcji.  
* **`V1`–`V28`** – 28 kolumn będących wynikami transformaty PCA (ang. *Principal Component Analysis*) zastosowanej do oryginalnego zestawu cech. Dzięki temu:
   - dane wrażliwe (np. dane o kliencie, o sprzedawcy czy geolokalizacja) nie są bezpośrednio ujawnione,  
   - w efekcie mamy pewien „skompresowany” opis transakcji, ukrywający pierwotne informacje.  
* **`Amount`** – kwota danej transakcji (bez standaryzacji w oryginalnym pliku).  
* **`Class`** – etykieta klasy: `0` (transakcja legalna/normalna) lub `1` (transakcja fraudowa/oszukańcza).

> **Uwaga**: Wskutek transformacji PCA, kolumny `V1`–`V28` mają zwykle wartości, które przypominają liczby o rozkładzie zbliżonym do gaussowskiego (często skupione wokół zera), choć specyficzny kształt rozkładu nie musi być idealnie normalny.

---

**Charakterystyka najważniejszych atrybutów**

1. Atrybut czasu (`Time`)

- Podaje odstęp (w sekundach) między pierwszą zarejestrowaną transakcją a każdą kolejną.  
- Czas często bywa wykorzystywany do identyfikowania pewnych wzorców np. pór dnia, w których transakcje fraudowe zdarzają się częściej.  
- Jednocześnie bywa, że modele nie potrzebują tej kolumny (lub traktują ją w inny sposób: np. rozbijają na interwały czasowe).

2. Atrybut kwoty (`Amount`)

- Reprezentuje wielkość (wartość) transakcji w jednostce monetarnej (brak jawnych informacji, czy to euro, dolary itp.).  
- Ma kluczowe znaczenie, gdyż nietypowe (bardzo wysokie) kwoty transakcji mogą wskazywać na próby oszustwa.  
- Zwykle w analizach praktycznych „Amount” jest dodatkowo skalowane (np. standaryzacją).

3. Etykiety fraudowe (`Class`)

- **0** = transakcja uznana za legalną,  
- **1** = transakcja wykryta (lub potwierdzona po fakcie) jako oszustwo.  
- Najczęściej bada się metryki jakości modeli, które próbują „przewidzieć” wystąpienie oszustwa dla nowej transakcji.

---

**Specyfika i główne wyzwania**

1. Skrajnie nierównomierny rozkład klas

- Transakcje oszukańcze stanowią jedynie ok. **0,172%** wszystkich przykładów – czyli około 492 przypadków fraudowych na ponad 284 tys. wierszy.

2. Dane już w formie zanonimizowanej / przetransformowanej

- Z jednej strony chroni to prywatność – nie mamy dostępu do wrażliwych informacji (np. ID karty, lokalizacji).  
- Z drugiej zaś utrudnia to interpretację cech (kolumn `V1`–`V28`), ponieważ nie znamy ich oryginalnego znaczenia.

3. Potencjalna stronniczość w czasie

- Zbiór obejmuje wyłącznie krótkie okno czasowe (ok. 2 dni).  
- W rzeczywistych systemach antyfraudowych dane zbierane są w sposób ciągły i obejmują różne okresy, co może powodować **concept drift** (zmieniające się wzorce oszustw).  

4. Rozkład wartości „Amount”

- W transakcjach kartowych występuje szeroki zakres kwot: od bardzo małych (np. kilka jednostek waluty) po bardzo duże (kilka tysięcy).  
- Często wartości te są skośne (ang. *skewed*); długim „ogonem” w kierunku wyższych kwot.  

---

**Podsumowanie**

Zbiór **creditcard.csv**:

1. **Jest unikatowym zasobem** – rzadko instytucje finansowe udostępniają prawdziwe dane transakcyjne.  
2. **Stanowi wyzwanie** – ekstremalnie nierównomierny rozkład klas (fraud vs. normalne transakcje) i brak informacji o oryginalnych cechach wprost.  
3. **Jest punktem odniesienia** – służy społeczności naukowej i praktykom do testowania oraz porównywania różnych metod, zarówno nadzorowanych, jak i nienadzorowanych, w kontekście wykrywania anomalii i fraudów.  
4. **Ograniczenia** – stosunkowo krótki przedział czasowy, brak szerszej metryki kontekstowej (np. informacji o geolokalizacji, typie sprzedawcy), konieczność radzenia sobie z dynamicznie zmieniającymi się schematami oszustw w rzeczywistości.

**W praktycznych projektach** analitycy i inżynierowie danych traktują ten zbiór jako „pierwszy poligon” doświadczalny, by następnie przenieść wypracowane wnioski do systemów produkcyjnych, w których często dostępne są bardziej zróżnicowane i bogate w kontekst dane (ale też objęte ostrzejszymi regulacjami prawnymi).

---

Wykorzystaj metody oparte o uczenie nienadzorowane:  
- **Isolation Forest**  
- **Local Outlier Factor (LOF)**  
- **One-Class SVM**

do wykrycia wartości odstających.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import time

In [2]:
pd.set_option('display.max_columns', None)        # pokazuj wszystkie kolumny
pd.set_option('display.expand_frame_repr', False) # nie łam wierszy na kilka linii

In [3]:
df = pd.read_csv("./Data/creditcard.csv")

# Utworzenie 5% próbki (z zachowaniem losowości powtarzalnej dzięki random_state), aby zmniejszyć czas obliczeń
df = df.sample(frac=0.05, random_state=42)

# Przygotowanie danych (df z 'Class' jako etykietą: 1 - fraud, 0 - normalny)
X = df.drop('Class', axis=1)
y_true = df['Class']

# Skalowanie
X_scaled = StandardScaler().fit_transform(X)

In [4]:
# Lista metod detekcji anomalii
models = {
    'Isolation Forest': IsolationForest(n_estimators=100, contamination='auto', random_state=42),
    'Local Outlier Factor': LocalOutlierFactor(n_neighbors=20, contamination='auto', novelty=True),
    'One-Class SVM': OneClassSVM(kernel='rbf', gamma='auto')
}

In [5]:
results = []

# Obliczamy liczbę anomalii i normalnych obserwacji w całym zbiorze
num_anomalies = sum(y_true == 1)
num_normal = sum(y_true == 0)

for name, model in models.items():
    # Pomiar czasu
    start_time = time.time()
    
    # Dopasowanie modelu do danych
    model.fit(X_scaled)
    
    # Przewidywanie: -1 = anomalia, 1 = normalny
    y_pred = model.predict(X_scaled)
    y_pred_binary = (y_pred == -1).astype(int)  # 1 = anomalia, 0 = normal <-- mapowanie, aby sprawdzić, czy znaleźliśmy
    
    end_time = time.time()
    train_time = end_time - start_time  # czas uczenia i predykcji
    
    # Metryki
    acc = accuracy_score(y_true, y_pred_binary)
    prec = precision_score(y_true, y_pred_binary, zero_division=0)
    rec = recall_score(y_true, y_pred_binary)
    f1 = f1_score(y_true, y_pred_binary)
    roc = roc_auc_score(y_true, y_pred_binary)
    
    results.append({
        'method': name,
        'train_time': train_time,
        'num_anomalies': num_anomalies,
        'num_normal': num_normal,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc
    })

In [6]:
# Tabela wyników
results_df = pd.DataFrame(results)
print(results_df)

                 method  train_time  num_anomalies  num_normal  accuracy  precision    recall        f1   roc_auc
0      Isolation Forest    0.226539             21       14219  0.963483   0.030132  0.761905  0.057971  0.862843
1  Local Outlier Factor    0.873322             21       14219  0.949087   0.021739  0.761905  0.042272  0.855634
2         One-Class SVM   16.083228             21       14219  0.501475   0.002949  1.000000  0.005882  0.750369


# Wnioski praktyczne

1. **Trade-off między *precision* a *recall***  
   - Zarówno Isolation Forest, jak i LOF mają niezły *recall* (~76%), ale bardzo niską precyzję (poniżej 3%).  
   - One-Class SVM znajduje wszystkie anomalie (recall=100%), ale niemal wszystko uznaje za anomalię, co jest nieakceptowalne w produkcyjnych warunkach (skrajnie niska precyzja).

2. **Liczba fałszywych alarmów (false positives)**  
   - Niska *precision* oznacza dużą liczbę normalnych transakcji błędnie oznaczanych jako oszustwo. W realnych systemach to ogromne koszty operacyjne (manualna weryfikacja, blokady kont etc.).

3. **Wpływ doboru hiperparametrów**  
   - Wyniki można poprawić np. strojąc parametr `contamination` w Isolation Forest czy w LOF, bądź regularyzację i jądro w One-Class SVM.  
   - Wysoka *recall* kosztem *precision* może być czasem akceptowalna w kontekście dalszej manualnej weryfikacji (jeśli wolimy więcej fałszywych alarmów niż przepuszczenie faktycznego fraudu).

4. **Czas trenowania**  
   - One-Class SVM jest zdecydowanie najwolniejszy (16 s vs ułamki sekundy dla pozostałych metod). Dla większych próbek i w kontekście systemów on-line ten aspekt może być niekorzystny. (czas dotyczy określonego komputera :) )

5. **Podsumowanie**  
   - Przy tak nierównomiernych danych (fraudy to ułamek procenta) modele nienadzorowane często generują wiele fałszywych alarmów.  
   - W praktyce stosuje się różne sposoby (m.in. tuning, kalibrację, łączenie z metodami nadzorowanymi, cechy kontekstowe) w celu poprawy *precision* przy zachowaniu rozsądnego *recall*.

**Ogólnie**: Dla ekstremalnie niezbalansowanego zadania wykrywania anomalii w płatnościach kartowych należy zawsze zważyć, co jest priorytetem – minimalizowanie utraconych fraudów czy ograniczanie liczby fałszywych alarmów. Każdy z tych modeli ma swoje zalety i wady, natomiast bez dalszego strojenia hiperparametrów wyniki sugerują, że zarówno Isolation Forest, jak i LOF mają podobne charakterystyki w wykrywaniu transakcji podejrzanych (przyzwoity recall, niska precision), a One-Class SVM jest znacznie bardziej skrajny w swoim podejściu (rejestruje wszystkie anomalia kosztem zasypania systemu fałszywymi alarmami).